<a href="https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI-Week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [4]:
from datasets import load_dataset
from huggingface_hub import login

login()

In [3]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset

# Load a reproducible sample from the approved warehouse.
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

df = pd.DataFrame(list(ds.take(10000)))

# Keep only fields available at prediction time.
feature_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_avg_position",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
]

features = df[feature_columns].copy()

# Numeric conversion.
numeric_features = [
    "gsc_impressions",
    "gsc_avg_position",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
]

for col in numeric_features:
    features[col] = pd.to_numeric(features[col], errors="coerce")

# Missing numeric values -> 0.
features[numeric_features] = features[numeric_features].fillna(0)

# Boolean fields -> 0/1.
boolean_features = [
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
]

for col in boolean_features:
    features[col] = features[col].fillna(False).astype(int)

print("Rows:", len(features))
print("Feature columns:")
print(features.columns.tolist())

features.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows: 10000
Feature columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_avg_position', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_avg_position,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,1,1,1,0,30,3.833333,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,1,1,1,0,5,71.600000,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,1,1,0,1,34.000000,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,1,1,1,0,6,23.333333,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,1,1,1,0,5,17.800000,0,0,0,0,0,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
feature_notes = pd.DataFrame([
    ["gsc_impressions", "Search exposure", "0", "Yes"],
    ["gsc_avg_position", "Observed average search position", "0", "Yes"],
    ["sessions_organic", "Organic traffic sessions", "0", "Yes"],
    ["sessions_direct", "Direct traffic sessions", "0", "Yes"],
    ["sessions_referral", "Referral traffic sessions", "0", "Yes"],
    ["sessions_social", "Social traffic sessions", "0", "Yes"],
    ["sessions_paid", "Paid traffic sessions", "0", "Yes"],
    ["sessions_ai", "AI-source traffic sessions", "0", "Yes"],
    ["client_has_gsc", "Whether GSC data is available for the client", "0/False", "Yes"],
    ["client_has_ga4", "Whether GA4 data is available for the client", "0/False", "Yes"],
], columns=[
    "feature",
    "meaning",
    "missing_handling",
    "available_before_prediction"
])

print(feature_notes.to_string(index=False))

          feature                                      meaning missing_handling available_before_prediction
  gsc_impressions                              Search exposure                0                         Yes
 gsc_avg_position             Observed average search position                0                         Yes
 sessions_organic                     Organic traffic sessions                0                         Yes
  sessions_direct                      Direct traffic sessions                0                         Yes
sessions_referral                    Referral traffic sessions                0                         Yes
  sessions_social                      Social traffic sessions                0                         Yes
    sessions_paid                        Paid traffic sessions                0                         Yes
      sessions_ai                   AI-source traffic sessions                0                         Yes
   client_has_gsc Whether GS

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# Explicitly identify fields that must not be model features.

target_components = {
    "gsc_clicks": "Direct click outcome; would leak a CTR/click target.",
    "CTR": "Derived from clicks and impressions; target-derived.",
    "trend_direction": "Can encode the outcome being predicted in the earlier task.",
    "trend_pct": "Outcome-derived trend information.",
}

available_columns = set(df.columns)

print("LEAKAGE CHECK")
print("=" * 60)

for column, reason in target_components.items():
    present = column in available_columns
    print(f"{column:20} present={present} | {reason}")

# Check that our feature vector contains none of the prohibited fields.
prohibited = set(target_components)
feature_leaks = sorted(set(features.columns) & prohibited)

print("\nFeatures used:")
print(features.columns.tolist())

print("\nPotential leakage columns in feature vector:")
print(feature_leaks)

assert not feature_leaks, f"Leakage detected: {feature_leaks}"

print("\nRESULT: No explicitly prohibited target-derived fields are in the feature vector.")

LEAKAGE CHECK
gsc_clicks           present=True | Direct click outcome; would leak a CTR/click target.
CTR                  present=False | Derived from clicks and impressions; target-derived.
trend_direction      present=False | Can encode the outcome being predicted in the earlier task.
trend_pct            present=False | Outcome-derived trend information.

Features used:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_avg_position', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai']

Potential leakage columns in feature vector:
[]

RESULT: No explicitly prohibited target-derived fields are in the feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [7]:
excluded_fields = pd.DataFrame([
    ["gsc_clicks", "Direct click outcome; would leak a click/CTR target."],
    ["gsc_sum_position", "Redundant with average position and can reconstruct position with exposure."],
    ["ga4_total_engagement_sec", "Post-click engagement; inappropriate for a pre-click CTR target."],
    ["ga4_engaged_sessions", "Post-click engagement outcome."],
    ["trend_direction", "Derived outcome information; not available safely as an independent predictor."],
    ["trend_pct", "Derived trend outcome information."],
    ["content_hash_id", "Identifier rather than a behavioral feature; avoids memorization of individual content IDs."],
    ["client_hash_id", "Identifier/group key rather than a predictive feature; avoids client memorization."],
], columns=["excluded_field", "reason"])

print(excluded_fields.to_string(index=False))
print("\nExcluded fields:", len(excluded_fields))

          excluded_field                                                                                      reason
              gsc_clicks                                        Direct click outcome; would leak a click/CTR target.
        gsc_sum_position                 Redundant with average position and can reconstruct position with exposure.
ga4_total_engagement_sec                            Post-click engagement; inappropriate for a pre-click CTR target.
    ga4_engaged_sessions                                                              Post-click engagement outcome.
         trend_direction              Derived outcome information; not available safely as an independent predictor.
               trend_pct                                                          Derived trend outcome information.
         content_hash_id Identifier rather than a behavioral feature; avoids memorization of individual content IDs.
          client_hash_id          Identifier/group key rather th

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.